# 02 – Data Cleaning
Cleans the SIMD 2020v2 indicators and joins the Data Zone (2011) Urban Rural Classification 2020,
based on findings from `01_data_exploration.ipynb`. Outputs are saved to `data/processed/`.

## 1. Load data and select columns
Keep identifiers, population, income deprivation and the nine travel time columns.
Health, education, crime and housing indicators are outside the scope of this project.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

simd_file = next(RAW.glob("*indicators*.xlsx"))
ur_file = next(RAW.glob("datazone2011_urban_rural*.csv"))

simd = pd.read_excel(simd_file, sheet_name="Data")
ur = pd.read_csv(ur_file)

In [2]:
travel_cols = [c for c in simd.columns if c.startswith(("drive_", "PT_"))]
keep = ["Data_Zone", "Intermediate_Zone", "Council_area", "Total_population", "Income_rate"] + travel_cols

simd = simd[keep]
print(simd.shape)

(6976, 14)


In [3]:
as_number = pd.to_numeric(simd["Income_rate"], errors="coerce")
not_numeric = simd.loc[as_number.isna(), "Income_rate"]

print("Non-numeric values:", len(not_numeric))
print(not_numeric.value_counts())

Non-numeric values: 3
Income_rate
*    3
Name: count, dtype: int64


**Result:** `Income_rate` contains 3 suppressed values ("*"). These are converted to missing
so the column becomes numeric. The 3 zones stay in the travel analysis and are excluded only
from deprivation comparisons.

In [4]:
simd["Income_rate"] = pd.to_numeric(simd["Income_rate"], errors="coerce")

print(simd["Income_rate"].dtype, "| missing:", simd["Income_rate"].isna().sum())
simd["Income_rate"].describe()

float64 | missing: 3


count    6973.000000
mean        0.122900
std         0.096091
min         0.000000
25%         0.040000
50%         0.100000
75%         0.180000
max         0.590000
Name: Income_rate, dtype: float64

## 2. Add the Urban Rural classification
Join the 6-fold classification by data zone, add readable labels, and create a simpler
3-group version (Urban / Small Towns / Rural) for clearer comparisons.

In [5]:
rows_before = len(simd)

simd = simd.merge(
    ur[["DataZone", "UrbanRural6fold2020"]],
    left_on="Data_Zone", right_on="DataZone",
    how="left", validate="one_to_one"
).drop(columns="DataZone")

print("Rows:", rows_before, "->", len(simd))
print("Missing classification:", simd["UrbanRural6fold2020"].isna().sum())

Rows: 6976 -> 6976
Missing classification: 0


In [6]:
ur_labels = {1: "Large Urban Areas", 2: "Other Urban Areas", 3: "Accessible Small Towns",
             4: "Remote Small Towns", 5: "Accessible Rural", 6: "Remote Rural"}
ur_groups = {1: "Urban", 2: "Urban", 3: "Small Towns", 4: "Small Towns", 5: "Rural", 6: "Rural"}

simd["UR_class"] = simd["UrbanRural6fold2020"].map(ur_labels)
simd["UR_group"] = simd["UrbanRural6fold2020"].map(ur_groups)

simd.groupby(["UrbanRural6fold2020", "UR_class", "UR_group"]).size()

UrbanRural6fold2020  UR_class                UR_group   
1                    Large Urban Areas       Urban          2496
2                    Other Urban Areas       Urban          2459
3                    Accessible Small Towns  Small Towns     613
4                    Remote Small Towns      Small Towns     202
5                    Accessible Rural        Rural           805
6                    Remote Rural            Rural           401
dtype: int64

**Result:** all 6,976 zones matched (one-to-one join, no rows lost). Each code maps to the
correct label and group, with counts identical to exploration.

## 3. Create access gap measures
For each service (GP, post office, retail centre):
- **gap (minutes)** = public transport time − drive time
- **ratio** = public transport time ÷ drive time (how many times longer)
The gap shows the extra time in absolute terms; the ratio shows the relative disadvantage.

In [7]:
services = {"GP": "GP", "post": "post", "retail": "retail"}

for name in services:
    simd[f"gap_{name}"] = simd[f"PT_{name}"] - simd[f"drive_{name}"]
    simd[f"ratio_{name}"] = simd[f"PT_{name}"] / simd[f"drive_{name}"]

gap_cols = [c for c in simd.columns if c.startswith(("gap_", "ratio_"))]
simd[gap_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
gap_GP,6976.0,6.897089,4.077876,-25.393277,4.131552,5.905032,8.566162,39.828706
ratio_GP,6976.0,3.134457,0.757336,0.568307,2.585961,3.065593,3.612537,6.911894
gap_post,6976.0,5.944877,3.309990,-1.388629,3.678740,5.127199,7.284370,28.496225
ratio_post,6976.0,3.163557,0.745366,0.918513,2.624564,3.115686,3.616599,7.560048
gap_retail,6976.0,8.204881,5.703488,-35.256231,4.749171,6.961338,10.089364,115.212765
ratio_retail,6976.0,2.761553,0.693092,0.417074,2.259972,2.682337,3.169743,7.201198


In [8]:
services = ["GP", "post", "retail"]

In [9]:
for name in ["GP", "post", "retail"]:
    neg = simd[simd[f"gap_{name}"] < 0]
    print(f"{name}: {len(neg)} zones with negative gap")

simd.loc[simd["gap_GP"] < -5,
         ["Data_Zone", "Council_area", "UR_class", "drive_GP", "PT_GP", "gap_GP"]].sort_values("gap_GP")

GP: 2 zones with negative gap
post: 1 zones with negative gap
retail: 2 zones with negative gap


,Data_Zone,Council_area,UR_class,drive_GP,PT_GP,gap_GP
781,S01007287,Argyll and Bute,Remote Rural,58.822601,33.429324,-25.393277
4000,S01010506,Highland,Remote Rural,38.899624,22.438536,-16.461087


**Result:** median public transport journeys are about 3× longer than driving for all three services
(extra 5–7 minutes). Only 5 zone-service combinations have negative gaps (public transport faster);
the largest are Remote Rural zones in Argyll and Bute and Highland, likely where passenger ferry or
public transport routes are shorter than the road route. These are kept and flagged.

## 4. Flag unusual values, validate and save
- `flag_pt_retail_capped`: PT_retail = 190 (likely a cap; true time is at least 190 minutes)
- `flag_negative_gap`: public transport faster than driving for any service
Flags allow these zones to be included or excluded in analysis without deleting data.

In [10]:
simd["flag_pt_retail_capped"] = simd["PT_retail"] >= 190
simd["flag_negative_gap"] = (simd[["gap_GP", "gap_post", "gap_retail"]] < 0).any(axis=1)

print(simd[["flag_pt_retail_capped", "flag_negative_gap"]].sum())

flag_pt_retail_capped    5
flag_negative_gap        3
dtype: int64


In [11]:
assert len(simd) == 6976, "Row count changed"
assert simd["Data_Zone"].is_unique, "Duplicate data zones"
assert simd[travel_cols].notna().all().all(), "Missing travel times"

PROCESSED.mkdir(parents=True, exist_ok=True)
simd.to_csv(PROCESSED / "access_clean.csv", index=False)

check = pd.read_csv(PROCESSED / "access_clean.csv")
print("Saved:", check.shape)

Saved: (6976, 25)


## Summary
Saved `data/processed/access_clean.csv`: 6,976 data zones × 25 columns.
- Kept identifiers, population, income deprivation and nine travel time columns.
- `Income_rate` converted to numeric; 3 suppressed values set to missing.
- Joined the Urban Rural Classification 2020 (one-to-one, all zones matched) with labels and a 3-group version.
- Added gap (minutes) and ratio measures for GP, post office and retail centre.
- Flagged 5 island zones with capped PT_retail (190 min) and 3 zones with negative gaps. No rows removed.